# FIFA 22 Complete Player Dataset - Machine Learning Analysis
## Group: FIFA Analytics | Members: Muhammad Mudassir, Abdul Latif, Asad Ullah | Section: BsBa 6A
---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, f1_score, classification_report, confusion_matrix
)

# Regression Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor,
    AdaBoostRegressor, ExtraTreesRegressor, BaggingRegressor
)
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
import xgboost as xgb
import lightgbm as lgb

# Classification Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    AdaBoostClassifier, ExtraTreesClassifier, BaggingClassifier
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

print('All libraries imported successfully!')

## 2. Load & Explore Data

In [ ]:
# Load dataset - update path as needed
df = pd.read_csv('players_22.csv', low_memory=False)
print('Shape:', df.shape)
df.head(3)

In [ ]:
print('Columns:', df.columns.tolist())
print('\nData Types:\n', df.dtypes.value_counts())
print('\nMissing Values (top 20):')
print(df.isnull().sum().sort_values(ascending=False).head(20))

In [ ]:
df[['value_eur', 'wage_eur', 'age', 'overall', 'potential']].describe()

## 3. Data Preprocessing & Feature Engineering

In [ ]:
# Select relevant features
features = [
    'age', 'overall', 'potential', 'wage_eur', 'release_clause_eur',
    'pace', 'shooting', 'passing', 'dribbling', 'defending', 'physic',
    'height_cm', 'weight_kg', 'international_reputation',
    'weak_foot', 'skill_moves'
]

target_reg = 'value_eur'
target_cls = 'player_positions'

# Create working copy
df_work = df[features + [target_reg, target_cls]].copy()

# Drop rows with missing target
df_work.dropna(subset=[target_reg, target_cls], inplace=True)

# Fill numeric NaN with median
for col in df_work.select_dtypes(include='number').columns:
    df_work[col].fillna(df_work[col].median(), inplace=True)

print('After cleaning shape:', df_work.shape)

In [ ]:
# Feature Engineering
df_work['potential_gap'] = df_work['potential'] - df_work['overall']
df_work['wage_to_value'] = df_work['wage_eur'] / (df_work['value_eur'] + 1)
df_work['age_group'] = pd.cut(df_work['age'], bins=[15, 21, 27, 33, 50],
                               labels=['Young', 'Prime', 'Experienced', 'Veteran'])

# Encode age_group
le_age = LabelEncoder()
df_work['age_group_enc'] = le_age.fit_transform(df_work['age_group'])

features_extended = features + ['potential_gap', 'wage_to_value', 'age_group_enc']
print('Features after engineering:', len(features_extended))

In [ ]:
# Map player positions to 4 categories
def map_position(pos):
    pos = str(pos).upper()
    gk = ['GK']
    defenders = ['CB', 'LB', 'RB', 'LWB', 'RWB', 'SW']
    midfielders = ['CM', 'CAM', 'CDM', 'LM', 'RM', 'DM']
    attackers = ['ST', 'CF', 'LW', 'RW', 'SS', 'LS', 'RS']
    for p in gk:
        if p in pos: return 'Goalkeeper'
    for p in defenders:
        if p in pos: return 'Defender'
    for p in midfielders:
        if p in pos: return 'Midfielder'
    for p in attackers:
        if p in pos: return 'Attacker'
    return 'Midfielder'  # default

df_work['position_cat'] = df_work[target_cls].apply(map_position)
print(df_work['position_cat'].value_counts())

## 4. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('FIFA 22 Player Data - EDA', fontsize=16, fontweight='bold')

# Value distribution
axes[0, 0].hist(np.log1p(df_work['value_eur']), bins=50, color='steelblue', edgecolor='black')
axes[0, 0].set_title('Player Market Value (log scale)')
axes[0, 0].set_xlabel('log(Value in EUR)')

# Overall rating distribution
axes[0, 1].hist(df_work['overall'], bins=30, color='coral', edgecolor='black')
axes[0, 1].set_title('Overall Rating Distribution')
axes[0, 1].set_xlabel('Overall Rating')

# Position category counts
pos_counts = df_work['position_cat'].value_counts()
axes[0, 2].bar(pos_counts.index, pos_counts.values, color=['gold', 'lightgreen', 'tomato', 'skyblue'])
axes[0, 2].set_title('Player Position Distribution')
axes[0, 2].set_ylabel('Count')

# Age vs Value
axes[1, 0].scatter(df_work['age'], np.log1p(df_work['value_eur']), alpha=0.3, color='purple')
axes[1, 0].set_title('Age vs Market Value')
axes[1, 0].set_xlabel('Age')
axes[1, 0].set_ylabel('log(Value EUR)')

# Overall vs Value
axes[1, 1].scatter(df_work['overall'], np.log1p(df_work['value_eur']), alpha=0.3, color='green')
axes[1, 1].set_title('Overall Rating vs Market Value')
axes[1, 1].set_xlabel('Overall Rating')
axes[1, 1].set_ylabel('log(Value EUR)')

# Correlation heatmap (top features)
corr_cols = ['overall', 'potential', 'age', 'wage_eur', 'pace', 'shooting', 'passing', 'value_eur']
corr = df_work[corr_cols].corr()
im = axes[1, 2].imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
axes[1, 2].set_xticks(range(len(corr_cols)))
axes[1, 2].set_yticks(range(len(corr_cols)))
axes[1, 2].set_xticklabels([c[:6] for c in corr_cols], rotation=45)
axes[1, 2].set_yticklabels([c[:6] for c in corr_cols])
axes[1, 2].set_title('Correlation Heatmap')
plt.colorbar(im, ax=axes[1, 2])

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Regression Task - Predicting Player Market Value (value_eur)

In [ ]:
# Prepare regression data
X_reg = df_work[features_extended].copy()
y_reg = np.log1p(df_work['value_eur'])  # log transform for normality

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

scaler_r = StandardScaler()
X_train_r_scaled = scaler_r.fit_transform(X_train_r)
X_test_r_scaled = scaler_r.transform(X_test_r)

print('Train size:', X_train_r.shape, '| Test size:', X_test_r.shape)

In [ ]:
# Define all regression models
reg_models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Lasso Regression': Lasso(alpha=0.01),
    'ElasticNet': ElasticNet(alpha=0.01, l1_ratio=0.5),
    'Decision Tree': DecisionTreeRegressor(max_depth=10, random_state=42),
    'KNN Regressor': KNeighborsRegressor(n_neighbors=5),
    'SVR': SVR(kernel='rbf', C=10),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42),
    'Bagging': BaggingRegressor(n_estimators=50, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'AdaBoost': AdaBoostRegressor(n_estimators=100, random_state=42),
    'XGBoost': xgb.XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1,
                                  random_state=42, verbosity=0),
    'LightGBM': lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05,
                                    random_state=42, verbose=-1),
}

reg_results = []

for name, model in reg_models.items():
    if name in ['Linear Regression', 'Ridge Regression', 'Lasso Regression',
                'ElasticNet', 'KNN Regressor', 'SVR']:
        model.fit(X_train_r_scaled, y_train_r)
        y_pred = model.predict(X_test_r_scaled)
    else:
        model.fit(X_train_r, y_train_r)
        y_pred = model.predict(X_test_r)

    rmse = np.sqrt(mean_squared_error(y_test_r, y_pred))
    mae = mean_absolute_error(y_test_r, y_pred)
    r2 = r2_score(y_test_r, y_pred)

    reg_results.append({'Model': name, 'RMSE': round(rmse, 4),
                        'MAE': round(mae, 4), 'R2 Score': round(r2, 4)})
    print(f'{name:<25} | RMSE: {rmse:.4f} | MAE: {mae:.4f} | R2: {r2:.4f}')

reg_df = pd.DataFrame(reg_results).sort_values('R2 Score', ascending=False)
print('\n--- Regression Results Sorted by R2 ---')
print(reg_df.to_string(index=False))

In [ ]:
# Visualize Regression Model Comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Regression Model Comparison', fontsize=14, fontweight='bold')

colors = ['green' if r == reg_df['R2 Score'].max() else 'steelblue' for r in reg_df['R2 Score']]

axes[0].barh(reg_df['Model'], reg_df['R2 Score'], color=colors)
axes[0].set_title('R² Score (Higher is Better)')
axes[0].set_xlabel('R² Score')
axes[0].axvline(0.9, color='red', linestyle='--', label='0.90 threshold')
axes[0].legend()

colors2 = ['green' if r == reg_df['RMSE'].min() else 'coral' for r in reg_df['RMSE']]
axes[1].barh(reg_df['Model'], reg_df['RMSE'], color=colors2)
axes[1].set_title('RMSE (Lower is Better)')
axes[1].set_xlabel('RMSE')

colors3 = ['green' if r == reg_df['MAE'].min() else 'orange' for r in reg_df['MAE']]
axes[2].barh(reg_df['Model'], reg_df['MAE'], color=colors3)
axes[2].set_title('MAE (Lower is Better)')
axes[2].set_xlabel('MAE')

plt.tight_layout()
plt.savefig('regression_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature Importance - XGBoost Regression
best_reg_model = xgb.XGBRegressor(n_estimators=200, max_depth=6,
                                    learning_rate=0.1, random_state=42, verbosity=0)
best_reg_model.fit(X_train_r, y_train_r)
importances = best_reg_model.feature_importances_
feat_imp = pd.Series(importances, index=features_extended).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
feat_imp.head(15).plot(kind='barh', color='steelblue')
plt.title('Top 15 Feature Importances (XGBoost - Regression)', fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('regression_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Classification Task - Predicting Player Position Category

In [ ]:
# Prepare classification data
le_pos = LabelEncoder()
y_cls = le_pos.fit_transform(df_work['position_cat'])
X_cls = df_work[features_extended].copy()

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

scaler_c = StandardScaler()
X_train_c_scaled = scaler_c.fit_transform(X_train_c)
X_test_c_scaled = scaler_c.transform(X_test_c)

print('Classes:', le_pos.classes_)
print('Train size:', X_train_c.shape, '| Test size:', X_test_c.shape)

In [ ]:
# Define all classification models
cls_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'KNN Classifier': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes': GaussianNB(),
    'SVM (RBF)': SVC(kernel='rbf', C=10, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Extra Trees': ExtraTreesClassifier(n_estimators=100, random_state=42),
    'Bagging': BaggingClassifier(n_estimators=50, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=42, algorithm='SAMME'),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                              random_state=42, use_label_encoder=False, eval_metric='mlogloss', verbosity=0),
    'LightGBM': LGBMClassifier(n_estimators=200, learning_rate=0.05,
                                random_state=42, verbose=-1),
}

cls_results = []

for name, model in cls_models.items():
    if name in ['Logistic Regression', 'KNN Classifier', 'Naive Bayes', 'SVM (RBF)']:
        model.fit(X_train_c_scaled, y_train_c)
        y_pred = model.predict(X_test_c_scaled)
    else:
        model.fit(X_train_c, y_train_c)
        y_pred = model.predict(X_test_c)

    acc = accuracy_score(y_test_c, y_pred)
    f1_macro = f1_score(y_test_c, y_pred, average='macro')
    f1_weighted = f1_score(y_test_c, y_pred, average='weighted')

    cls_results.append({'Model': name, 'Accuracy': round(acc, 4),
                        'F1 Macro': round(f1_macro, 4), 'F1 Weighted': round(f1_weighted, 4)})
    print(f'{name:<25} | Acc: {acc:.4f} | F1-Macro: {f1_macro:.4f} | F1-Weighted: {f1_weighted:.4f}')

cls_df = pd.DataFrame(cls_results).sort_values('Accuracy', ascending=False)
print('\n--- Classification Results Sorted by Accuracy ---')
print(cls_df.to_string(index=False))

In [ ]:
# Visualize Classification Model Comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Classification Model Comparison', fontsize=14, fontweight='bold')

colors_acc = ['green' if a == cls_df['Accuracy'].max() else 'steelblue' for a in cls_df['Accuracy']]
axes[0].barh(cls_df['Model'], cls_df['Accuracy'], color=colors_acc)
axes[0].set_title('Accuracy (Higher is Better)')
axes[0].set_xlabel('Accuracy')
axes[0].axvline(0.85, color='red', linestyle='--', label='85% threshold')
axes[0].legend()

colors_f1m = ['green' if f == cls_df['F1 Macro'].max() else 'coral' for f in cls_df['F1 Macro']]
axes[1].barh(cls_df['Model'], cls_df['F1 Macro'], color=colors_f1m)
axes[1].set_title('F1 Score - Macro')
axes[1].set_xlabel('F1 Macro')

colors_f1w = ['green' if f == cls_df['F1 Weighted'].max() else 'orange' for f in cls_df['F1 Weighted']]
axes[2].barh(cls_df['Model'], cls_df['F1 Weighted'], color=colors_f1w)
axes[2].set_title('F1 Score - Weighted')
axes[2].set_xlabel('F1 Weighted')

plt.tight_layout()
plt.savefig('classification_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion Matrix - Best Classification Model (XGBoost)
best_cls = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                          random_state=42, use_label_encoder=False,
                          eval_metric='mlogloss', verbosity=0)
best_cls.fit(X_train_c, y_train_c)
y_pred_best = best_cls.predict(X_test_c)

cm = confusion_matrix(y_test_c, y_pred_best)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_pos.classes_, yticklabels=le_pos.classes_)
plt.title('Confusion Matrix - XGBoost (Classification)', fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nClassification Report:')
print(classification_report(y_test_c, y_pred_best, target_names=le_pos.classes_))

In [ ]:
# Feature Importance - XGBoost Classification
imp_cls = pd.Series(best_cls.feature_importances_, index=features_extended).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
imp_cls.head(15).plot(kind='barh', color='coral')
plt.title('Top 15 Feature Importances (XGBoost - Classification)', fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('classification_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Hyperparameter Tuning - Best Models

In [ ]:
# Tune XGBoost Regressor
param_grid_reg = {
    'n_estimators': [100, 200],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1, 0.2]
}

xgb_reg = xgb.XGBRegressor(random_state=42, verbosity=0)
grid_reg = GridSearchCV(xgb_reg, param_grid_reg, cv=3, scoring='r2', n_jobs=-1)
grid_reg.fit(X_train_r, y_train_r)

print('Best Params (Regression XGBoost):', grid_reg.best_params_)
print('Best CV R2 Score:', round(grid_reg.best_score_, 4))

y_pred_tuned = grid_reg.predict(X_test_r)
print('Tuned Test R2:', round(r2_score(y_test_r, y_pred_tuned), 4))
print('Tuned Test RMSE:', round(np.sqrt(mean_squared_error(y_test_r, y_pred_tuned)), 4))

In [ ]:
# Tune XGBoost Classifier
param_grid_cls = {
    'n_estimators': [100, 200],
    'max_depth': [4, 6],
    'learning_rate': [0.05, 0.1]
}

xgb_cls = XGBClassifier(random_state=42, use_label_encoder=False,
                          eval_metric='mlogloss', verbosity=0)
grid_cls = GridSearchCV(xgb_cls, param_grid_cls, cv=3, scoring='accuracy', n_jobs=-1)
grid_cls.fit(X_train_c, y_train_c)

print('Best Params (Classification XGBoost):', grid_cls.best_params_)
print('Best CV Accuracy:', round(grid_cls.best_score_, 4))

y_pred_cls_tuned = grid_cls.predict(X_test_c)
print('Tuned Test Accuracy:', round(accuracy_score(y_test_c, y_pred_cls_tuned), 4))
print('Tuned Test F1 (Weighted):', round(f1_score(y_test_c, y_pred_cls_tuned, average='weighted'), 4))

## 8. Summary & Business Recommendations

In [ ]:
print('='*60)
print('REGRESSION TASK SUMMARY - Player Market Value Prediction')
print('='*60)
print(reg_df.to_string(index=False))
print(f'\nBest Model: {reg_df.iloc[0]["Model"]} | R2: {reg_df.iloc[0]["R2 Score"]}')

print('\n' + '='*60)
print('CLASSIFICATION TASK SUMMARY - Player Position Prediction')
print('='*60)
print(cls_df.to_string(index=False))
print(f'\nBest Model: {cls_df.iloc[0]["Model"]} | Acc: {cls_df.iloc[0]["Accuracy"]}')

print('\n' + '='*60)
print('BUSINESS RECOMMENDATIONS')
print('='*60)
print('''
1. Transfer Market Valuation:
   - Use the XGBoost regression model to estimate fair market value
     before negotiations. Prevents overpaying in transfer windows.

2. Player Scouting:
   - The classification model can auto-assign young academy players
     to optimal positions based on their physical/skill attributes.

3. Youth Development:
   - The 'potential_gap' feature highlights undervalued young players
     with high growth potential — ideal for budget club strategies.

4. Squad Optimization:
   - Combine both models to find the best lineup within a budget:
     predict each candidate's value AND best position.
''')